# SurviveCity v2 — Kaggle T4 training (single-GPU bound, all v1-pattern fixes)

Trains the v2 GRPO LoRA on a SINGLE T4 (~15 GB VRAM). Although Kaggle exposes
2 T4s when you select 'T4 x2', this notebook explicitly binds to GPU 0 only —
bitsandbytes 4-bit + DataParallel breaks dual-T4 (v1 hit this hard).

## Before you run

1. **Settings → Accelerator → T4 x2** (we use only GPU 0)
2. **Settings → Internet → On**
3. **Add-ons → Secrets**:
   - `HF_TOKEN` — write-scoped HuggingFace token
   - `GITHUB_TOKEN` — GitHub PAT with `repo` scope
4. **Settings → Persistence → Files only**

Run cells 0 → 8 in order. Each cell fails fast on misconfiguration so you
don't burn 2 hours discovering a setup error mid-training. Cell 9 is the
long run (~3-4 h).

## Cell 0 — Configuration + env locks (run FIRST, before any torch import)

Sets `CUDA_VISIBLE_DEVICES=0`, `GIT_TERMINAL_PROMPT=0`, etc. before any later cell
imports torch — the torch import caches the visible-GPU list, so env changes after
import are too late.

In [ ]:
# === SurviveCity v2 — Kaggle T4 training ===
import os, sys

# --- Hub / repo config ---
HUB_MODEL_ID = "noanya/zombiee-v2"
REPO_URL     = "https://github.com/SirjanSingh/zombiee.git"
REPO_BRANCH  = "master"
REPO_DIR     = "/kaggle/working/zombiee"
V2_DIR       = os.path.join(REPO_DIR, "v2")
OUTPUT_DIR   = "/kaggle/working/checkpoints"
MODEL_NAME   = "Qwen/Qwen2.5-3B-Instruct"   # NOT the unsloth-bnb-4bit prequant — bf16-baked-in breaks T4

# --- Training config (T4-tuned) ---
MAX_STEPS               = 15
SAVE_STEPS              = 1
SAVE_TOTAL_LIMIT        = 15
PER_DEVICE_BATCH_SIZE   = 4
NUM_GENERATIONS         = 4
GRAD_ACCUM_STEPS        = 4
MAX_COMPLETION_LENGTH   = 256
MAX_PROMPT_LENGTH       = 1024
LORA_R                  = 16
LORA_ALPHA              = 32
LR                      = 1e-5

# --- Environment locks ---
# Bind to GPU 0 only. Kaggle's dual-T4 + bitsandbytes 4-bit + DataParallel breaks (CUBLAS).
os.environ["CUDA_VISIBLE_DEVICES"]      = "0"
# Reduce allocator fragmentation during long GRPO runs
os.environ["PYTORCH_CUDA_ALLOC_CONF"]   = "expandable_segments:True,max_split_size_mb:128"
# Suppress wandb prompts (we use tensorboard)
os.environ["WANDB_DISABLED"]            = "true"
os.environ["TOKENIZERS_PARALLELISM"]    = "false"
# Block git from EVER trying to prompt for credentials
os.environ["GIT_TERMINAL_PROMPT"]       = "0"
os.environ["GIT_ASKPASS"]               = "/bin/echo"

# Defensive: drop any pre-imported torch so cell 1's install + cell 2's import re-read env
for mod in list(sys.modules):
    if mod == "torch" or mod.startswith("torch."):
        del sys.modules[mod]

print(f"Config loaded. HUB_MODEL_ID = {HUB_MODEL_ID}")
print(f"Plan: {MAX_STEPS} steps, save every {SAVE_STEPS} step → {MAX_STEPS} checkpoints on Hub")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}  (single-GPU bind)")

## Cell 1 — Pin cu121 torch FIRST, then everything else

Pinning torch from the cu121 index BEFORE installing accelerate/peft/trl prevents
pip's resolver from silently replacing CUDA torch with PyPI's CPU-only wheel.

In [ ]:
import subprocess, sys

def run_pip(args, label=None):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    if label:
        print(f">>> {label}")
    subprocess.check_call(cmd)

# 1. Pin cu121 torch FIRST so subsequent installs don't replace it with CPU torch
run_pip(
    ["install", "--quiet", "--no-cache-dir", "--upgrade-strategy=only-if-needed",
     "torch==2.5.1", "torchvision==0.20.1",
     "--index-url", "https://download.pytorch.org/whl/cu121"],
    label="Installing torch 2.5.1+cu121 (pinned)",
)

# 2. Core training stack
run_pip(
    ["install", "--quiet", "--no-cache-dir", "--upgrade-strategy=only-if-needed",
     "transformers==4.46.3", "accelerate==1.1.1", "peft==0.13.2",
     "datasets==3.1.0", "trl==0.15.2"],
    label="Installing transformers/peft/trl pin set",
)

# 3. 4-bit + trl-cascade dep + plotting
run_pip(
    ["install", "--quiet", "--no-cache-dir", "--upgrade-strategy=only-if-needed",
     "bitsandbytes>=0.43", "mergekit",
     "matplotlib>=3.8", "tensorboard", "huggingface_hub>=0.25",
     "hf_transfer", "pydantic>=2.0"],
    label="Installing bnb / mergekit / logging deps",
)

print("\n>>> All packages installed.")

## Cell 2 — Verify torch is CUDA + only 1 GPU visible (fail fast)

Hard-asserts both. If this cell raises, fix the cause before going further — every
later cell would fail too.

In [ ]:
import torch

print(f"torch version:   {torch.__version__}")
print(f"CUDA build:      {torch.version.cuda}")
print(f"CUDA available:  {torch.cuda.is_available()}")
print(f"GPU count:       {torch.cuda.device_count()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "torch.cuda.is_available() is False. Either pip replaced cu121 torch "
        "with CPU-only, OR Kaggle accelerator is wrong. Settings → Accelerator → T4 x2."
    )
if torch.cuda.device_count() != 1:
    raise RuntimeError(
        f"Expected 1 visible GPU, got {torch.cuda.device_count()}. "
        "Cell 0 didn't run before torch was imported. Kernel → Restart, then re-run from cell 0."
    )

free_gb  = torch.cuda.mem_get_info(0)[0] / 1e9
total_gb = torch.cuda.mem_get_info(0)[1] / 1e9
cap      = torch.cuda.get_device_capability(0)
print(f"GPU 0:           {torch.cuda.get_device_name(0)}  cc={cap[0]}.{cap[1]}  bf16={torch.cuda.is_bf16_supported()}")
print(f"VRAM free:       {free_gb:.1f} / {total_gb:.1f} GB")
print("\nGPU binding confirmed.")

## Cell 3 — Apply dep-cascade workarounds

Fixes from Dockerfile.dgx history:
1. Uninstall torchao (peft >= 0.13's strict version check)
2. Patch transformers/utils/hub.py with TRANSFORMERS_CACHE alias
3. Stub llm_blender (real package conflicts with our pinned stack)

In [ ]:
import subprocess, sys, os, sysconfig, shutil

# 1) Uninstall torchao if pulled in (peft >=0.13's strict version check fires otherwise)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
               check=False, capture_output=True)

# 2) Patch transformers/utils/hub.py with TRANSFORMERS_CACHE alias
import transformers.utils.hub as hub_mod
hub_path = hub_mod.__file__
with open(hub_path, "r") as f:
    src = f.read()
if "TRANSFORMERS_CACHE" not in src:
    with open(hub_path, "a") as f:
        f.write(
            "\n# Legacy alias (Kaggle notebook)\n"
            'TRANSFORMERS_CACHE = HF_HUB_CACHE if "HF_HUB_CACHE" in dir() else "~/.cache/huggingface/hub"\n'
        )
    print(f">>> Patched {hub_path}")
else:
    print(f">>> {hub_path} already patched")

# 3) Stub out llm_blender (real package conflicts with our pinned transformers)
site_packages = sysconfig.get_paths()["purelib"]
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "llm-blender"],
               check=False, capture_output=True)

stub_dir = os.path.join(site_packages, "llm_blender")
if os.path.exists(stub_dir):
    shutil.rmtree(stub_dir)
os.makedirs(stub_dir, exist_ok=True)

with open(os.path.join(stub_dir, "__init__.py"), "w") as f:
    f.write('''"""Stub llm_blender — Kaggle v2 GRPO does not use judges."""

class Blender:
    def __init__(self, *args, **kwargs):
        raise RuntimeError("llm_blender stubbed")
    def loadranker(self, *args, **kwargs): pass
    def loadfuser(self, *args, **kwargs): pass
    def rank(self, *args, **kwargs): raise RuntimeError("llm_blender stubbed")
    def fuse(self, *args, **kwargs): raise RuntimeError("llm_blender stubbed")
''')
print(f">>> Wrote llm_blender stub to {stub_dir}/__init__.py")

# Force re-import
for mod in list(sys.modules):
    if mod == "llm_blender" or mod.startswith("llm_blender."):
        del sys.modules[mod]

print("\n>>> Workarounds applied.")

## Cell 4 — Full import smoke test

Mirror of `v2/Dockerfile.dgx`'s smoke test. If this passes, all training-time imports
are guaranteed to work.

In [ ]:
import importlib.util, torch, transformers, peft, datasets, accelerate, bitsandbytes, huggingface_hub, trl
import mergekit, llm_blender
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.utils.hub import TRANSFORMERS_CACHE
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training, PeftModel
from trl import GRPOTrainer, GRPOConfig

ta = importlib.util.find_spec("torchao")
print(f"torch        {torch.__version__}  (cuda={torch.cuda.is_available()})")
print(f"transformers {transformers.__version__}")
print(f"peft         {peft.__version__}")
print(f"trl          {trl.__version__}")
print(f"bitsandbytes {bitsandbytes.__version__}")
print(f"torchao      {'absent (intended)' if ta is None else 'PRESENT (unexpected)'}")
assert ta is None, "torchao should be uninstalled"
assert torch.cuda.is_available(), "GPU not detected — restart kernel from cell 0"
assert torch.cuda.device_count() == 1, f"expected 1 GPU, got {torch.cuda.device_count()}"
print("\nFULL TRAINING IMPORT CHAIN OK")

## Cell 5 — Clone private repo with GITHUB_TOKEN

Auth via `http.extraheader` (token not in argv → safe in `ps aux`). Set
`GIT_TERMINAL_PROMPT=0` AND `GIT_ASKPASS=/bin/echo` in the subprocess env so git
fails fast instead of hanging on a credential prompt.

In [ ]:
import os, subprocess, base64, getpass

# 1) Read GitHub PAT — Kaggle Secret first, then env var, then interactive prompt
gh = None
try:
    from kaggle_secrets import UserSecretsClient
    gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
if not gh:
    gh = os.environ.get("GITHUB_TOKEN")
if not gh:
    gh = getpass.getpass("GitHub PAT (scope: repo): ")
if not gh:
    raise RuntimeError("No GitHub token available. Add GITHUB_TOKEN to Kaggle Secrets or set the env var.")

# 2) Clean any prior partial clone
subprocess.run(["rm", "-rf", REPO_DIR], check=False)

# 3) Auth via http.extraheader (token never appears in argv → safe in `ps aux`)
b64 = base64.b64encode(f"x-access-token:{gh}".encode()).decode()
auth = f"http.https://github.com/.extraheader=Authorization: Basic {b64}"

# 4) Clone with extraheader auth AND no-prompt env vars (set in cell 0)
clone_env = os.environ.copy()
clone_env["GIT_TERMINAL_PROMPT"] = "0"
clone_env["GIT_ASKPASS"]         = "/bin/echo"

try:
    subprocess.run(
        ["git", "-c", auth, "clone", "--depth", "1",
         "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
        env=clone_env, check=True, capture_output=True, text=True,
    )
except subprocess.CalledProcessError as e:
    msg = ((e.stdout or "") + (e.stderr or "")).replace(gh, "***").replace(b64, "***")
    raise RuntimeError(f"git clone failed (rc={e.returncode}):\n{msg}") from None

# 5) Wipe token from memory
del gh, b64, auth

# 6) Show what we got
log = subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-3"],
                     capture_output=True, text=True).stdout
print(">>> Latest commits:\n" + log)

os.chdir(V2_DIR)
print(f">>> cwd: {os.getcwd()}")
print(">>> v2 contents:")
for entry in sorted(os.listdir(".")):
    print("    ", entry)

## Cell 6 — HF login + ensure hub repo exists

Uses `huggingface_hub.login(add_to_git_credential=True)` for both API and git-push
auth. Pre-creates `noanya/zombiee-v2` so the first save doesn't race with create_repo.
Tiny write test confirms WRITE access before training launches.

In [ ]:
import os, getpass
from huggingface_hub import login, HfApi

# Read HF token — Kaggle Secrets, env var, then prompt
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    for name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            hf_token = client.get_secret(name)
            if hf_token:
                break
        except Exception:
            continue
except Exception:
    pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if not hf_token:
    hf_token = getpass.getpass("HF token (write scope): ")
if not hf_token:
    raise RuntimeError("No HF token available. Add HF_TOKEN to Kaggle Secrets.")

# Login (sets credential helper for both Hub API and git push)
login(token=hf_token, add_to_git_credential=True)
os.environ["HUGGINGFACE_TOKEN"] = hf_token
os.environ["HF_TOKEN"]          = hf_token

# Pre-create the hub repo so the first save doesn't race with create_repo
api = HfApi()
try:
    api.create_repo(repo_id=HUB_MODEL_ID, exist_ok=True, private=True)
    print(f">>> Hub repo ready: https://huggingface.co/{HUB_MODEL_ID}")
except Exception as e:
    print(f">>> create_repo warning (ok if exists): {e}")

# Tiny write test
import tempfile
with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as f:
    f.write("kaggle-auth-test\n")
    test_path = f.name
try:
    api.upload_file(path_or_fileobj=test_path, path_in_repo=".kaggle_auth_test.txt",
                    repo_id=HUB_MODEL_ID, commit_message="kaggle auth test")
    api.delete_file(path_in_repo=".kaggle_auth_test.txt",
                    repo_id=HUB_MODEL_ID, commit_message="cleanup auth test")
    print(f">>> WRITE access to {HUB_MODEL_ID} confirmed")
finally:
    os.remove(test_path)

## Cell 7 — Detect existing checkpoints on Hub (for resume)

If you re-run after a Kaggle disconnect, this picks up where training left off.

In [ ]:
# Wipe stale state from any previous failed run.
# We're starting fresh after the parse-failure bug fix, so any prior dud
# LoRA on Hub should NOT be resumed from.
import shutil, os

# Local: clean checkpoints/ dir
if os.path.isdir(OUTPUT_DIR):
    print(f">>> Wiping local {OUTPUT_DIR} (fresh start)")
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hub: report any stale files from a previous (broken) run.
# They'll be OVERWRITTEN by the first save of the new run.
files_on_hub = api.list_repo_files(HUB_MODEL_ID)
adapter_files = [f for f in files_on_hub if f.startswith("adapter_") or f == "training_args.bin"]
if adapter_files:
    print(f">>> Hub has {len(adapter_files)} stale adapter file(s) from previous run:")
    for f in adapter_files:
        print(f"     {f}")
    print(">>> They'll be OVERWRITTEN by the first save of the new run (no manual delete needed).")
else:
    print(">>> Hub is clean.")

# We do NOT use --resume-from-checkpoint here. Always start fresh after a bug fix.
resume_arg = []
print("\n>>> resume_arg =", resume_arg, "(fresh training)")


## Cell 8 — Local v2 env smoke test

Imports the v2 env and runs one full episode with random actions. Catches broken env
code before we burn 3 hours on training.

In [ ]:
import sys, random
sys.path.insert(0, V2_DIR)

from survivecity_v2_env.env import SurviveCityV2Env

env = SurviveCityV2Env()
obs = env.reset(seed=42)
print(f">>> Reset OK: step={obs['step_count']}, reward={obs['reward']:.4f}, "
      f"infected={obs['metadata']['starting_infected']}")
assert 0.01 <= obs["reward"] <= 0.99

actions = ["move_up", "move_down", "move_left", "move_right",
           "eat", "drink", "wait", "pickup"]
steps = 0
while not obs.get("done", False) and steps < 350:
    aid = obs["metadata"]["current_agent_id"]
    sc  = obs["step_count"]
    if sc in (30, 60, 90):
        action = {"agent_id": aid, "action_type": "vote_lockout",
                  "vote_target": random.choice([0, 1, 2, 3, 4])}
    else:
        action = {"agent_id": aid, "action_type": random.choice(actions)}
    obs = env.step(action)
    assert 0.01 <= obs["reward"] <= 0.99
    steps += 1

meta = obs.get("metadata", {})
print(f">>> Episode complete: {steps} actions, done={obs['done']}, "
      f"alive={meta.get('n_alive')}, healthy={meta.get('n_healthy_alive')}, "
      f"bites={len(meta.get('bite_history', []))}, "
      f"postmortems={len(meta.get('postmortems', []))}")
print(">>> v2 env smoke test PASSED")

## Cell 9 — Launch GRPO training

Runs `training.train` as a subprocess. Stream output live so you can watch progress.
Subprocess env explicitly carries `CUDA_VISIBLE_DEVICES=0`, alloc config, and HF token.

In [ ]:
import subprocess, sys, os

os.makedirs(OUTPUT_DIR, exist_ok=True)

cmd = [
    sys.executable, "-m", "training.train",
    "--model-name", MODEL_NAME,
    "--max-steps", str(MAX_STEPS),
    "--save-steps", str(SAVE_STEPS),
    "--save-total-limit", str(SAVE_TOTAL_LIMIT),
    "--per-device-batch-size", str(PER_DEVICE_BATCH_SIZE),
    "--num-generations", str(NUM_GENERATIONS),
    "--grad-accum-steps", str(GRAD_ACCUM_STEPS),
    "--max-completion-length", str(MAX_COMPLETION_LENGTH),
    "--max-prompt-length", str(MAX_PROMPT_LENGTH),
    "--lora-r", str(LORA_R),
    "--lora-alpha", str(LORA_ALPHA),
    "--lr", str(LR),
    "--max-seq-length", "4096",
    "--vram-reserve-gb", "0",
    "--push-to-hub",
    "--hub-model-id", HUB_MODEL_ID,
    "--report-to", "tensorboard",
    "--output-dir", OUTPUT_DIR,
] + resume_arg

# Force single-GPU + correct allocator + token in subprocess env
proc_env = os.environ.copy()
proc_env["CUDA_VISIBLE_DEVICES"]    = "0"
proc_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
proc_env["WANDB_DISABLED"]          = "true"
proc_env["HF_TOKEN"]                = os.environ["HF_TOKEN"]
proc_env["HUGGINGFACE_TOKEN"]       = os.environ["HUGGINGFACE_TOKEN"]

print(">>> Launching:", " ".join(cmd))
print("=" * 70)

proc = subprocess.Popen(cmd, cwd=V2_DIR, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1, env=proc_env)
try:
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    print("=" * 70)
    print(f">>> Training exited with code {rc}")
    if rc != 0:
        raise RuntimeError(f"Training failed (exit {rc})")
except KeyboardInterrupt:
    print("\nInterrupted; sending SIGTERM. Latest checkpoint is on the Hub.")
    proc.terminate()
    proc.wait()

## Cell 10 — Safety-net upload (rescue any local-only artefacts)

If training's `--push-to-hub` had a transient network failure, this catches anything
still local in OUTPUT_DIR and uploads it.

In [ ]:
import os
from huggingface_hub import HfApi

api = HfApi()
if os.path.isdir(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    print(f">>> Uploading {OUTPUT_DIR} → {HUB_MODEL_ID}")
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HUB_MODEL_ID,
        commit_message="kaggle final upload (safety net)",
        ignore_patterns=["_resume/*", "*.log"],
    )
    print(">>> Done")
else:
    print(f">>> Nothing to upload at {OUTPUT_DIR}")

## Cell 11 — Verify all checkpoints landed on Hub

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files(HUB_MODEL_ID)

checkpoints = sorted({f.split("/")[0] for f in files if f.startswith("checkpoint-")})
print(f">>> Files on {HUB_MODEL_ID}: {len(files)} total")
print(f">>> Checkpoints found: {len(checkpoints)} → {checkpoints}")

if len(checkpoints) >= MAX_STEPS:
    print(f">>> All {MAX_STEPS} checkpoints pushed.")
elif checkpoints:
    print(f">>> Only {len(checkpoints)} checkpoints — training stopped early.")
else:
    print(">>> NO checkpoint subdirs.")

## Cell 12 — TensorBoard inline (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/checkpoints/runs